# 01 — Feature tables, DuckDB views, and engagement

Run `python -m pipeline.run` from the repo root first so `clean/` is populated.


In [ ]:
import sys; sys.path.append('..')
import duckdb
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from pipeline import io as cio, extract

PALETTE = ['#1f77b4', '#ff7f0e', '#d62728', '#17becf']
sns.set_theme(style='whitegrid')
sns.set_palette(PALETTE)
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=PALETTE)
plt.rcParams['figure.dpi'] = 110

CLEANED = cio.CLEANED_DIR
YEARS = extract.YEARS
sorted(p.name for p in CLEANED.glob('*'))

## Feature tables built by `pipeline/load.py`


In [ ]:
repo_feat = cio.read_alloc_repo_year_features()
user_feat = cio.read_alloc_user_year_features()
print('alloc_repo_year_features:', repo_feat.shape)
print('alloc_user_year_features:', user_feat.shape)
repo_feat.head()

In [ ]:
print('repo-year rows per year')
print(repo_feat.groupby('year').size())
print('\nuser-year rows per year')
print(user_feat.groupby('year').size())

## DuckDB sanity check


In [ ]:
con = duckdb.connect(str(CLEANED / 'nersc.duckdb'), read_only=True)
con.execute('SHOW TABLES').fetchdf()

In [ ]:
con.execute('SELECT * FROM v_repo_year_metrics LIMIT 5').fetchdf()

In [ ]:
con.execute('''SELECT * FROM v_pi_succession
               ORDER BY n_pi_changes DESC LIMIT 10''').fetchdf()

## Active repo vs repos without ever burning hours.

In [ ]:
active = con.execute('''
    SELECT year, COUNT(DISTINCT repo) AS n_active
    FROM v_active_repos
    GROUP BY year ORDER BY year
''').fetchdf()
active

## Users active in multiple funding offices in the same year.



In [ ]:
multi_office = con.execute('''
    SELECT year, COUNT(*) AS n_users_in_multiple_offices
    FROM v_user_office_span
    WHERE n_offices_touched >= 2
    GROUP BY year ORDER BY year
''').fetchdf()
multi_office

## Engagement tiers - peripheral, casual, active, or core over time 

In [ ]:
eng = con.execute('SELECT * FROM v_engagement_tier ORDER BY year, user_id').fetchdf()
tier_order = ['peripheral', 'casual', 'active', 'core']
eng['tier'] = pd.Categorical(eng['tier'], categories=tier_order, ordered=True)
palette = globals().get('PALETTE', ['#1f77b4', '#ff7f0e', '#d62728', '#17becf'])

tier_year = eng.groupby(['year', 'tier'], observed=False).size().reset_index(name='n_user_years')
tier_pivot = (tier_year.pivot(index='year', columns='tier', values='n_user_years')
              .reindex(columns=tier_order).fillna(0))
tier_share = tier_pivot.div(tier_pivot.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(8.5, 4.5))
tier_share[tier_order].plot(kind='bar', stacked=True, ax=ax, color=palette)
ax.set_ylabel('share of user-years')
ax.set_title('Engagement tier mix by year')
ax.legend(title='tier', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()
tier_share.round(3)

## Engagement by office in 2026

In [ ]:
office_tier = con.execute('''
WITH user_office AS (
  SELECT DISTINCT user_id, year, office
  FROM allocations
  WHERE office IS NOT NULL
)
SELECT u.office, e.tier, COUNT(*) AS n_user_office_years
FROM user_office u
JOIN v_engagement_tier e USING (user_id, year)
WHERE u.year = 2026
GROUP BY u.office, e.tier
''').fetchdf()

office_totals = office_tier.groupby('office')['n_user_office_years'].sum().sort_values(ascending=False)
top_offices = office_totals.head(12).index
office_heat = (office_tier[office_tier['office'].isin(top_offices)]
               .pivot(index='office', columns='tier', values='n_user_office_years')
               .fillna(0))
office_heat = office_heat.reindex(columns=tier_order).fillna(0)
office_heat = office_heat.div(office_heat.sum(axis=1), axis=0)

plt.figure(figsize=(9, 5.5))
sns.heatmap(office_heat, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1)
plt.title('2026 engagement mix by office (share within office)')
plt.xlabel('tier'); plt.ylabel('office')
plt.tight_layout(); plt.show()
office_heat.round(3)

## Project-lead vs non-lead engagement - is the community lead-heavy or distributed


In [ ]:
pi_tier = (eng.groupby(['year', 'is_pi', 'tier']).size()
             .reset_index(name='n_user_years'))
pi_pivot = (pi_tier.pivot(index=['year', 'is_pi'], columns='tier', values='n_user_years')
            .fillna(0))
for col in tier_order:
    if col not in pi_pivot.columns:
        pi_pivot[col] = 0
pi_pivot = pi_pivot[tier_order]
pi_share = pi_pivot.div(pi_pivot.sum(axis=1), axis=0).reset_index()
pi_share['role'] = pi_share['is_pi'].map({True: 'Lead', False: 'Non-lead'})

fig, ax = plt.subplots(figsize=(8, 4.5))
for role, style in [('Lead', '-o'), ('Non-lead', '--s')]:
    sub = pi_share[pi_share['role'] == role]
    ax.plot(sub['year'], sub['core'], style, label=f'{role}: core share')
    ax.plot(sub['year'], sub['active'] + sub['core'], style.replace('o', '^').replace('s', 'd'),
            alpha=0.55, label=f'{role}: active+core share')
ax.set_ylim(0, 1)
ax.set_ylabel('share of user-years')
ax.set_title('Project-lead vs non-lead engagement over time')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

pi_2026 = pi_share[pi_share['year'] == 2026][['role'] + tier_order].set_index('role')
pi_2026

## Data invariants


In [ ]:
from pipeline.load import run_invariants
failures = run_invariants(con)
if failures:
    raise AssertionError(f'Invariants violated: {failures}')
print('All invariants pass:')
for name in sorted(failures.keys() if failures else [
    'alloc_has_negative_charged_cpu',
    'alloc_has_negative_charged_gpu',
    'pi_history_orphan',
    'repo_null_in_alloc',
    'user_id_null_in_alloc',
]):
    print(f'  ✓ {name}')